# Synthetic Paper Plots

Final, paper-oriented figures for the multivariate synthetic sweeps. This notebook intentionally keeps only the polished plotting pipeline, separate from exploratory summary notebooks.

Set HSIC_RUNS to your input sweep directory (default runs/multivariate_sweep). Set HSIC_OUTPUT_DIR to a new output folder to preserve existing figures. Run cells in order. Archived U-centered delta intervals are rebuilt from saved score variance and V using the current robust rule; negative U repetitions are retained. No nuisance refitting is needed. See SYNTHETIC_REPRODUCTION.md.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
from statistics import NormalDist
import math
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from load_experiment_results import load_experiment_rows
from paper_plot_utils import apply_axis_visibility, build_ci_frame, reconstruct_robust_delta_intervals, remove_legends, save_figure_variants, slugify, summarize_binary

# The paper embeds Liberation Sans. Register installed fonts explicitly so a
# stale Matplotlib cache cannot silently substitute DejaVu and change geometry.
from matplotlib import font_manager
for font_path in Path("/usr/share/fonts/truetype/liberation").glob("LiberationSans-*.ttf"):
    font_manager.fontManager.addfont(str(font_path))
font_manager.findfont(font_manager.FontProperties(family="Liberation Sans"), fallback_to_default=False)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = ["Liberation Sans"]
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "semibold",
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

BASE = Path(os.environ.get("HSIC_RUNS", "runs/multivariate_sweep"))
OUTPUT = Path(os.environ.get("HSIC_OUTPUT_DIR", "."))
FIGURE_DIR = OUTPUT / "figures/synthetic"
ALPHA = 0.05
Z975 = 1.959963984540054

FAVOURABILITY_PATTERN = os.environ.get("HSIC_SYNTHETIC_PATTERN", "favourability_search_updated_*")
SAMPLE_SIZE = 250  # Used by individual-panel helpers; the full grid now shows all sample sizes.
SIGNAL_SMOOTHNESS = 1.0
SIGNAL_SMOOTHNESS_VALUES = [SIGNAL_SMOOTHNESS, 1.5, 2.0]
GRID_SAMPLE_SIZES = [250, 500, 1000]
SINGLE_ROW_SAMPLE_SIZE = 500
SINGLE_ROW_SAMPLE_SIZES = GRID_SAMPLE_SIZES
RHO_MIN = 0.0
RHO_MAX = 0.5

SAVE_FIGURES = True
SAVE_FULL_FIGURE = True
SAVE_SINGLE_ROW_FIGURE = True
SAVE_PANEL_FIGURES = False
SAVE_FORMATS = ("png", "pdf")


In [ ]:
METHOD_ORDER = [
    "bootstrap_naive",
    "bootstrap_naive_u_centered",
    "bootstrap_debiased",
    "bootstrap_debiased_u_centered",
    "delta_naive_v",
    "delta_naive_v_loo",
    "delta_naive_u",
    "delta_naive_u_loo",
    "delta_debiased_v",
    "delta_debiased_v_loo",
    "delta_debiased_u",
    "delta_debiased_u_loo",
]

FAVOURABILITY_KEEP_COLUMNS = [
    "N",
    "rho",
    "f_smoothness",
    "eps_smoothness",
    "dimx",
    "dimy",
    "random_key",
    "population_oracle.hsic",
    "population_oracle.sqrt_hsic",
    "permutation.naive_hsic_u_p_value",
    "sqrt_hsic.oracle_hsic_v",
    "sqrt_hsic.oracle_hsic_u",
    "oracle.crossfit_hsic_v",
    "oracle.crossfit_hsic_u",
    "sqrt_hsic.naive_hsic_v",
    "sqrt_hsic.naive_hsic_u",
    "sqrt_hsic.debiased_hsic_v",
    "sqrt_hsic.debiased_hsic_u",
    "estimators.naive_hsic_u",
    "estimators.naive_hsic_v",
    "estimators.debiased_hsic_v",
    "config.ci_alpha",
    "estimators.debiased_hsic_u",
]
for method in METHOD_ORDER:
    FAVOURABILITY_KEEP_COLUMNS.extend(
        [
            f"intervals.{method}.ci_lower",
            f"intervals.{method}.ci_upper",
            f"intervals.{method}.sqrt_lower",
            f"intervals.{method}.sqrt_upper",
            f"intervals.{method}.q_val",
            f"intervals.{method}.radius",
            f"intervals.{method}.score_var",
            f"intervals.{method}.n",
        ]
    )
FAVOURABILITY_KEEP_COLUMNS = sorted(set(FAVOURABILITY_KEEP_COLUMNS))


def load_joint_favourability_runs(pattern: str = FAVOURABILITY_PATTERN) -> pd.DataFrame:
    rows = []
    for run_dir in sorted(BASE.glob(pattern)):
        loaded = load_experiment_rows(run_dir, include_raw=False, write_cache=False, keep_columns=FAVOURABILITY_KEEP_COLUMNS)
        for row in loaded:
            row = dict(row)
            row["source_run_name"] = run_dir.name
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    numeric_cols = [
        "N",
        "rho",
        "f_smoothness",
        "eps_smoothness",
        "random_key",
        "population_oracle.hsic",
        "population_oracle.sqrt_hsic",
        "permutation.naive_hsic_u_p_value",
        "intervals.bootstrap_naive.ci_lower",
        "intervals.bootstrap_naive.ci_upper",
        "intervals.bootstrap_debiased.ci_lower",
        "intervals.bootstrap_debiased.ci_upper",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    dedupe_cols = ["N", "rho", "f_smoothness", "eps_smoothness", "random_key"]
    if all(col in df.columns for col in dedupe_cols):
        df = (
            df.sort_values(["source_run_name", "run_name"])
            .drop_duplicates(dedupe_cols, keep="last")
            .reset_index(drop=True)
        )
    return df


archived_favourability = load_joint_favourability_runs()
if archived_favourability.empty:
    raise FileNotFoundError(f"No synthetic results under {BASE}")
if not np.isclose(pd.to_numeric(archived_favourability["config.ci_alpha"]), ALPHA).all():
    raise ValueError("Expected 95% source intervals; use a separate analysis for another confidence level")
favourability = reconstruct_robust_delta_intervals(archived_favourability)
print(f"Loaded {len(favourability):,} deduplicated favourability rows from {FAVOURABILITY_PATTERN!r}.")
if not favourability.empty:
    display(
        favourability[["N", "rho", "f_smoothness", "eps_smoothness"]]
        .drop_duplicates()
        .sort_values(["N", "f_smoothness", "eps_smoothness", "rho"])
        .reset_index(drop=True)
    )

if favourability.empty:
    raise FileNotFoundError(f"No synthetic results under {BASE}; see SYNTHETIC_REPRODUCTION.md")
TABLE_DIR = OUTPUT / "tables/synthetic"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
favourability.to_csv(TABLE_DIR / "analysis_input_rows.csv.gz", index=False)

## Synthetic Data-Generating Mechanism

One-off Matplotlib visual check matching the one-dimensional DGP from `toy_cross_fitting.ipynb`.


In [ ]:
# One-off visual check for the one-dimensional synthetic data-generating mechanism.
# This follows the DGP from toy_cross_fitting.ipynb, but uses Matplotlib for paper-style output.
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

DGP_SAMPLE_SIZE = 1000
DGP_RANDOM_KEY = 0
DGP_XMIN, DGP_XMAX = (-3, 3)
DGP_BASE_STD = 0.35
DGP_SIGNAL_STD = 1.0
DGP_F_MODES = 20
DGP_F_OMEGA0 = 1.5
DGP_EPS_MODES = 20
DGP_EPS_OMEGA0 = 0.5
DGP_PROBE_SIZE = 2000
DGP_GRID_SIZE = 900

DGP_PREVIEW_RHO = 0.5
DGP_PREVIEW_M_SMOOTHNESS = 1.0
DGP_PREVIEW_SIGMA_SMOOTHNESS = 0.75

DGP_FIG_WIDTH = 6.6
DGP_FIG_HEIGHT = 5.2
DGP_TITLE_SIZE = 11.0
DGP_AXIS_LABEL_SIZE = 12.0
DGP_TICK_LABEL_SIZE = 9.0
DGP_MARKER_SIZE = 7.5
DGP_MARKER_ALPHA = 0.42
DGP_LEFT = 0.075
DGP_RIGHT = 0.992
DGP_BOTTOM = 0.075
DGP_TOP = 0.935
DGP_HSPACE = 0.045


def fourier_series_raw(x, mode_idx, coef, phase, smoothness, omega0):
    x = jnp.asarray(x, dtype=jnp.float64)
    amps = coef / (mode_idx ** smoothness)
    return jnp.sum(amps * jnp.sin(omega0 * x[..., None] * mode_idx + phase), axis=-1)


def build_synthetic_dgp_1d(
    *,
    rho: float,
    m_smoothness: float,
    sigma_smoothness: float,
    sample_size: int = DGP_SAMPLE_SIZE,
    random_key: int = DGP_RANDOM_KEY,
) -> dict[str, np.ndarray]:
    key = jax.random.key(random_key)
    probe_x = jnp.linspace(DGP_XMIN, DGP_XMAX, DGP_PROBE_SIZE)

    f_mode_idx = jnp.arange(1, DGP_F_MODES + 1, dtype=jnp.float64)
    f_coef = jax.random.normal(jax.random.fold_in(key, 0), shape=(DGP_F_MODES,))
    f_phase = jax.random.uniform(
        jax.random.fold_in(key, 1), shape=(DGP_F_MODES,), minval=0.0, maxval=2.0 * jnp.pi
    )
    f_probe_scale = jnp.maximum(
        jnp.std(fourier_series_raw(probe_x, f_mode_idx, f_coef, f_phase, m_smoothness, DGP_F_OMEGA0)),
        1e-12,
    )

    def m_true(x):
        return DGP_SIGNAL_STD * fourier_series_raw(x, f_mode_idx, f_coef, f_phase, m_smoothness, DGP_F_OMEGA0) / f_probe_scale

    eps_mode_idx = jnp.arange(1, DGP_EPS_MODES + 1, dtype=jnp.float64)
    eps_coef = jax.random.normal(jax.random.fold_in(key, 2), shape=(DGP_EPS_MODES,))
    eps_phase = jax.random.uniform(
        jax.random.fold_in(key, 3), shape=(DGP_EPS_MODES,), minval=0.0, maxval=2.0 * jnp.pi
    )
    eps_probe_scale = jnp.maximum(
        jnp.std(
            fourier_series_raw(
                probe_x,
                eps_mode_idx,
                eps_coef,
                eps_phase,
                sigma_smoothness,
                DGP_EPS_OMEGA0,
            )
        ),
        1e-12,
    )

    def eps_shape(x):
        return fourier_series_raw(x, eps_mode_idx, eps_coef, eps_phase, sigma_smoothness, DGP_EPS_OMEGA0) / eps_probe_scale

    def eps_std_fn(x):
        return DGP_BASE_STD * jnp.exp(jnp.maximum(rho, 0.0) * eps_shape(x))

    def sample_noise(key, x, sample_shape=None):
        sigma = eps_std_fn(x)
        if sample_shape is None:
            sample_shape = jnp.shape(sigma)
        return sigma * jax.random.normal(key, shape=sample_shape)

    X = jax.random.uniform(jax.random.fold_in(key, 0), shape=(sample_size,), minval=DGP_XMIN, maxval=DGP_XMAX)
    Eps = sample_noise(jax.random.fold_in(key, 1), X)
    Mean = jax.vmap(m_true)(X)
    Y = Mean + Eps

    x_grid = jnp.linspace(DGP_XMIN, DGP_XMAX, DGP_GRID_SIZE)
    mean_grid = jax.vmap(m_true)(x_grid)
    sigma_grid = eps_std_fn(x_grid)
    order = jnp.argsort(X)

    return {
        "x": np.asarray(X[order]),
        "mean": np.asarray(Mean[order]),
        "eps": np.asarray(Eps[order]),
        "y": np.asarray(Y[order]),
        "x_grid": np.asarray(x_grid),
        "mean_grid": np.asarray(mean_grid),
        "sigma_grid": np.asarray(sigma_grid),
    }


def plot_synthetic_dgp_1d(
    *,
    rho: float = DGP_PREVIEW_RHO,
    m_smoothness: float = DGP_PREVIEW_M_SMOOTHNESS,
    sigma_smoothness: float = DGP_PREVIEW_SIGMA_SMOOTHNESS,
    include_sigma_smoothness: bool | None = None,
):
    if include_sigma_smoothness is None:
        include_sigma_smoothness = not np.isclose(rho, 0.0)
    data = build_synthetic_dgp_1d(
        rho=rho,
        m_smoothness=m_smoothness,
        sigma_smoothness=sigma_smoothness,
    )

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(DGP_FIG_WIDTH, DGP_FIG_HEIGHT),
        sharex=True,
        gridspec_kw={"hspace": DGP_HSPACE},
    )

    axes[0].plot(data["x_grid"], data["mean_grid"], color="#1d4ed8", linewidth=1.8)

    axes[1].fill_between(
        data["x_grid"],
        -2.0 * data["sigma_grid"],
        2.0 * data["sigma_grid"],
        color="#f97316",
        alpha=0.13,
        linewidth=0.0,
    )
    axes[1].plot(data["x_grid"], 2.0 * data["sigma_grid"], color="#7c2d12", linewidth=1.15, linestyle="--")
    axes[1].plot(data["x_grid"], -2.0 * data["sigma_grid"], color="#7c2d12", linewidth=1.15, linestyle="--")
    axes[1].scatter(data["x"], data["eps"], s=DGP_MARKER_SIZE, color="#f97316", alpha=DGP_MARKER_ALPHA, linewidths=0.0)

    axes[2].fill_between(
        data["x_grid"],
        data["mean_grid"] - 2.0 * data["sigma_grid"],
        data["mean_grid"] + 2.0 * data["sigma_grid"],
        color="#14b8a6",
        alpha=0.14,
        linewidth=0.0,
    )
    axes[2].plot(data["x_grid"], data["mean_grid"] + 2.0 * data["sigma_grid"], color="#0f766e", linewidth=1.15, linestyle="--")
    axes[2].plot(data["x_grid"], data["mean_grid"] - 2.0 * data["sigma_grid"], color="#0f766e", linewidth=1.15, linestyle="--")
    axes[2].scatter(data["x"], data["y"], s=DGP_MARKER_SIZE, color="#111827", alpha=0.34, linewidths=0.0)
    axes[2].plot(data["x_grid"], data["mean_grid"], color="#dc2626", linewidth=1.6)

    for ax, label in zip(axes, [r"$m(x)$", r"$\epsilon$", r"$Y$"]):
        ax.set_xlim(-3.1, 3.1)
        ax.set_ylabel(label, rotation=0, labelpad=14, fontsize=DGP_AXIS_LABEL_SIZE, va="center")
        ax.tick_params(axis="both", labelsize=DGP_TICK_LABEL_SIZE)
        ax.grid(True, axis="y", color="0.90", linewidth=0.8)
        ax.grid(False, axis="x")

    axes[-1].set_xlabel(r"$X$", fontsize=DGP_AXIS_LABEL_SIZE)
    title = rf"$\rho={rho:g},\ s_m={m_smoothness:g}"
    if include_sigma_smoothness:
        title += rf",\ s_\sigma={sigma_smoothness:g}"
    title += "$"
    fig.suptitle(title, y=0.975, fontsize=DGP_TITLE_SIZE, fontweight="semibold")
    fig.subplots_adjust(left=DGP_LEFT, right=DGP_RIGHT, bottom=DGP_BOTTOM, top=DGP_TOP)
    return fig


preview_dgp_fig = plot_synthetic_dgp_1d()
plt.show()


In [ ]:
dgp_saved_paths = []
DGP_SAVE_FIGURES = SAVE_FIGURES
DGP_RHOS = [0.0, 0.3, 0.5]
DGP_M_SMOOTHNESS = 1.0
DGP_SIGMA_SMOOTHNESS_VALUES = [0.25, 0.5, 0.75]


def _format_dgp_smoothness(value: float) -> str:
    return f"{float(value):g}"

if DGP_SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for rho_value in DGP_RHOS:
        sigma_values = [DGP_SIGMA_SMOOTHNESS_VALUES[-1]] if np.isclose(rho_value, 0.0) else DGP_SIGMA_SMOOTHNESS_VALUES
        for sigma_smoothness in sigma_values:
            include_sigma_smoothness = not np.isclose(rho_value, 0.0)
            fig = plot_synthetic_dgp_1d(
                rho=rho_value,
                m_smoothness=DGP_M_SMOOTHNESS,
                sigma_smoothness=sigma_smoothness,
                include_sigma_smoothness=include_sigma_smoothness,
            )
            stem = (
                f"synthetic_dgp_rho_{slugify(_format_dgp_smoothness(rho_value))}"
                f"_m_smoothness_{slugify(_format_dgp_smoothness(DGP_M_SMOOTHNESS))}"
            )
            if include_sigma_smoothness:
                stem += f"_sigma_smoothness_{slugify(_format_dgp_smoothness(sigma_smoothness))}"
            for fmt in SAVE_FORMATS:
                path = FIGURE_DIR / f"{stem}.{fmt}"
                fig.savefig(path, dpi=300, bbox_inches="tight")
                dgp_saved_paths.append(path)
            plt.close(fig)

print(f"Saved {len(dgp_saved_paths)} DGP figure files to {FIGURE_DIR}.")
if dgp_saved_paths:
    display(pd.DataFrame({"path": [str(path) for path in dgp_saved_paths]}))


In [ ]:
PROCEDURE_ORDER = ["bootstrap debiased", "permutation"]
PROCEDURE_COLORS = plt.get_cmap("tab10").colors
PROCEDURE_PALETTE = {"bootstrap debiased": "#2563eb", "permutation": "#f97316"}
PROCEDURE_LABELS = {"bootstrap debiased": "debiased bootstrap", "permutation": "full-sample cross-fit permutation"}
PROCEDURE_MARKERS = {"permutation": "o", "bootstrap debiased": "^"}
PROCEDURE_LINESTYLES = {"permutation": "-", "bootstrap debiased": "-."}


def wilson_interval(successes: int, total: int, z: float = Z975) -> tuple[float, float, float]:
    if total <= 0:
        return np.nan, np.nan, np.nan
    p_hat = successes / total
    denom = 1.0 + z**2 / total
    center = (p_hat + z**2 / (2.0 * total)) / denom
    half_width = z * math.sqrt((p_hat * (1.0 - p_hat) / total) + (z**2 / (4.0 * total**2))) / denom
    return p_hat, max(0.0, center - half_width), min(1.0, center + half_width)


def build_rejection_test_frame(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    base = df.loc[
        (df["rho"] >= RHO_MIN)
        & (df["rho"] <= RHO_MAX)
    ].copy()
    frames = []

    if "permutation.naive_hsic_u_p_value" in base.columns:
        block = base[["N", "rho", "f_smoothness", "eps_smoothness", "random_key", "permutation.naive_hsic_u_p_value"]].copy()
        block["procedure"] = "permutation"
        block["reject"] = block["permutation.naive_hsic_u_p_value"] < ALPHA
        block = block.dropna(subset=["reject"])
        frames.append(block[["procedure", "N", "rho", "f_smoothness", "eps_smoothness", "random_key", "reject"]])

    for procedure, prefix in [
        ("bootstrap debiased", "intervals.bootstrap_debiased"),
    ]:
        lower_col = f"{prefix}.ci_lower"
        upper_col = f"{prefix}.ci_upper"
        if lower_col not in base.columns or upper_col not in base.columns:
            continue
        block = base[["N", "rho", "f_smoothness", "eps_smoothness", "random_key", lower_col, upper_col]].copy()
        block["procedure"] = procedure
        block["lower"] = pd.to_numeric(block[lower_col], errors="coerce")
        block["upper"] = pd.to_numeric(block[upper_col], errors="coerce")
        block = block.dropna(subset=["lower", "upper"])
        block["reject"] = ~((block["lower"] <= 0.0) & (0.0 <= block["upper"]))
        frames.append(block[["procedure", "N", "rho", "f_smoothness", "eps_smoothness", "random_key", "reject"]])

    if not frames:
        return pd.DataFrame()

    tests = pd.concat(frames, ignore_index=True)
    tests["procedure"] = pd.Categorical(tests["procedure"], categories=PROCEDURE_ORDER, ordered=True)
    return tests.sort_values(["procedure", "N", "f_smoothness", "eps_smoothness", "rho", "random_key"]).reset_index(drop=True)


def summarize_rejection_probability(tests: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["procedure", "N", "rho", "f_smoothness", "eps_smoothness"]
    for key, block in tests.groupby(group_cols, dropna=False, observed=True):
        successes = int(block["reject"].sum())
        total = len(block)
        reject_rate, lower, upper = wilson_interval(successes, total)
        row = dict(zip(group_cols, key))
        row.update(
            {
                "runs": total,
                "rejects": successes,
                "reject_rate": reject_rate,
                "reject_rate_lower": lower,
                "reject_rate_upper": upper,
                "reject_rate_err_low": reject_rate - lower,
                "reject_rate_err_high": upper - reject_rate,
            }
        )
        rows.append(row)
    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary["procedure"] = pd.Categorical(summary["procedure"], categories=PROCEDURE_ORDER, ordered=True)
        summary = summary.sort_values(["f_smoothness", "N", "eps_smoothness", "rho", "procedure"]).reset_index(drop=True)
    return summary


rejection_tests = build_rejection_test_frame(favourability)
rejection_summary = summarize_rejection_probability(rejection_tests)
print(f"Built {len(rejection_tests):,} test rows and {len(rejection_summary):,} summarized rejection probabilities.")
display(rejection_summary.round(4))

rejection_summary.to_csv(TABLE_DIR / "rejection_summary.csv", index=False)

In [ ]:
# Manual layout knobs for the full rejection-probability grid.
# These are deliberately direct: edit them, rerun this cell, then rerun the preview/save cell.
FACET_LABEL_SIZE = 9.5  # Font size for facet labels: top s_epsilon titles and vertical s_m row labels.
AXIS_LABEL_SIZE = 9.5  # Font size for global x/y axis labels and individual-panel axis labels.
PANEL_TITLE_SIZE = 9.5  # Font size for individual panel titles when saving panels separately.

GRID_PANEL_WIDTH = 2.25  # Width, in inches, allocated to each facet column; lower makes the full grid less wide.
GRID_PANEL_HEIGHT = 2.25  # Height, in inches, allocated to each facet row; higher makes the full grid taller.
GRID_LEFT = 0.015  # Left edge of the subplot grid in figure coordinates; higher moves panels right.
GRID_RIGHT = 0.985  # Right edge of the subplot grid in figure coordinates; lower moves panels left/narrows grid.
GRID_BOTTOM = 0.08  # Bottom edge of the subplot grid; higher creates more room below panels.
GRID_TOP_WITH_LEGEND = 0.9  # Top edge of panels when the shared legend is shown.
GRID_TOP_NO_LEGEND = 0.91  # Top edge of panels when the shared legend is hidden.
GRID_WSPACE = 0.1  # Horizontal space between facet columns; higher spreads columns apart.
GRID_HSPACE = 0.1  # Vertical space between facet rows; higher spreads rows apart.

GRID_TITLE_Y = 0.985  # Vertical position of the main title; lower moves title toward panels.
GRID_LEGEND_Y = 0.965  # Vertical position of the shared legend; lower moves legend toward panels.
GRID_XLABEL_Y = 0.025  # Vertical position of the shared x-axis label.
GRID_YLABEL_OFFSET = 0.08  # Distance between left panel edge and global y-label; lower moves it closer to panels.
GRID_ROW_LABEL_OFFSET = 0.05  # Distance between left panel edge and vertical row labels; lower moves s_m labels closer.
GRID_COLUMN_TITLE_PAD = 3  # Padding, in points, between top-row s_epsilon titles and their axes.

# Separate layout knobs for the single-N rejection-probability row.
SINGLE_ROW_PANEL_WIDTH = 2.55
SINGLE_ROW_PANEL_HEIGHT = 2.35
SINGLE_ROW_LEFT = 0.085
SINGLE_ROW_RIGHT = 0.91
SINGLE_ROW_BOTTOM = 0.2
SINGLE_ROW_TOP = 0.73
SINGLE_ROW_WSPACE = 0.10
SINGLE_ROW_TITLE_Y = 0.985
SINGLE_ROW_LEGEND_Y = 0.935
SINGLE_ROW_XLABEL_Y = 0.035
SINGLE_ROW_YLABEL_X = 0.018


def _format_smoothness(value: float) -> str:
    return f"{float(value):g}"


def _plot_rejection_lines(ax, panel: pd.DataFrame, *, show_legend: bool = False) -> None:
    for procedure in PROCEDURE_ORDER:
        block = panel.loc[panel["procedure"] == procedure].sort_values("rho")
        if block.empty:
            continue
        yerr = np.vstack([block["reject_rate_err_low"].to_numpy(), block["reject_rate_err_high"].to_numpy()])
        ax.errorbar(
            block["rho"],
            block["reject_rate"],
            yerr=yerr,
            label=PROCEDURE_LABELS[procedure],
            color=PROCEDURE_PALETTE[procedure],
            marker=PROCEDURE_MARKERS[procedure],
            linestyle=PROCEDURE_LINESTYLES[procedure],
            linewidth=1.45,
            markersize=3.7,
            capsize=2.3,
            capthick=0.85,
            elinewidth=0.85,
            alpha=0.95,
        )
    ax.axhline(ALPHA, color="0.25", linestyle=":", linewidth=1.0, zorder=0)
    ax.set_ylim(-0.025, 1.025)
    rho_values = sorted(panel["rho"].dropna().unique())
    ax.set_xticks(rho_values)
    ax.set_yticks(np.linspace(0.0, 1.0, 6))
    ax.grid(True, axis="y", color="0.88", linewidth=0.8)
    ax.grid(False, axis="x")
    if show_legend:
        ax.legend(title=None, loc="best")


def plot_rejection_probability_grid(
    summary: pd.DataFrame,
    *,
    signal_smoothness: float = SIGNAL_SMOOTHNESS,
    sample_sizes: list[int] | tuple[int, ...] | None = GRID_SAMPLE_SIZES,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
    show_row_labels: bool = True,
    title_sample_size: int | None = None,
):
    plot_df = summary.loc[summary["f_smoothness"] == signal_smoothness].copy()
    if sample_sizes is None:
        n_values = sorted(plot_df["N"].dropna().unique())
    else:
        n_values = [int(value) for value in sample_sizes]
        plot_df = plot_df.loc[plot_df["N"].isin(n_values)].copy()
    eps_values = sorted(plot_df["eps_smoothness"].dropna().unique())

    fig, axes = plt.subplots(
        len(n_values),
        len(eps_values),
        figsize=(GRID_PANEL_WIDTH * len(eps_values), GRID_PANEL_HEIGHT * len(n_values)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    for row_idx, n_value in enumerate(n_values):
        for col_idx, eps_value in enumerate(eps_values):
            ax = axes[row_idx, col_idx]
            panel = plot_df.loc[
                (plot_df["N"] == n_value)
                & (plot_df["eps_smoothness"] == eps_value)
            ].copy()
            _plot_rejection_lines(ax, panel, show_legend=False)
            if row_idx == 0:
                ax.set_title(r"$s_\epsilon$ = " + _format_smoothness(eps_value), pad=GRID_COLUMN_TITLE_PAD, fontsize=FACET_LABEL_SIZE, fontweight="semibold")
            apply_axis_visibility(ax, include_x_axis=include_x_axis, include_y_axis=include_y_axis)
            show_x_tick_values = include_x_axis and row_idx == len(n_values) - 1
            show_y_tick_values = include_y_axis and col_idx == 0
            ax.tick_params(
                axis="x",
                which="both",
                bottom=show_x_tick_values,
                labelbottom=show_x_tick_values,
            )
            ax.tick_params(
                axis="y",
                which="both",
                left=show_y_tick_values,
                labelleft=show_y_tick_values,
            )

    if include_x_axis:
        fig.supxlabel(r"dependence strength $\rho$", y=GRID_XLABEL_Y, fontsize=AXIS_LABEL_SIZE)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if include_legend and handles:
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, GRID_LEGEND_Y),
            ncol=len(PROCEDURE_ORDER),
            title=None,
        )
        top = GRID_TOP_WITH_LEGEND
    else:
        remove_legends(fig)
        top = GRID_TOP_NO_LEGEND

    title = r"Rejection probability at 5% level ($s_m$ = " + _format_smoothness(signal_smoothness)
    if title_sample_size is not None:
        title += r", $N$ = " + str(int(title_sample_size))
    title += ")"
    fig.suptitle(title, y=GRID_TITLE_Y, fontsize=12.0, fontweight="semibold")
    fig.subplots_adjust(
        left=GRID_LEFT if include_y_axis else max(0.02, GRID_LEFT - GRID_YLABEL_OFFSET),
        right=GRID_RIGHT,
        bottom=GRID_BOTTOM if include_x_axis else max(0.02, GRID_BOTTOM - 0.04),
        top=top,
        wspace=GRID_WSPACE,
        hspace=GRID_HSPACE,
    )

    first_col_left = min(ax.get_position().x0 for ax in axes[:, 0])
    row_label_x = first_col_left - GRID_ROW_LABEL_OFFSET
    if include_y_axis:
        fig.text(
            first_col_left - GRID_YLABEL_OFFSET,
            0.5,
            "rejection probability",
            rotation=90,
            va="center",
            ha="center",
            fontsize=AXIS_LABEL_SIZE,
        )
    if show_row_labels:
        for row_idx, n_value in enumerate(n_values):
            row_pos = axes[row_idx, 0].get_position()
            fig.text(
                row_label_x,
                0.5 * (row_pos.y0 + row_pos.y1),
                r"$N$ = " + str(int(n_value)),
                rotation=90,
                va="center",
                ha="center",
                fontsize=FACET_LABEL_SIZE,
                fontweight="semibold",
            )
    return fig


def plot_rejection_probability_single_row(
    summary: pd.DataFrame,
    *,
    sample_size: int = SINGLE_ROW_SAMPLE_SIZE,
    signal_smoothness: float = SIGNAL_SMOOTHNESS,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
):
    plot_df = summary.loc[
        (summary["N"] == sample_size)
        & (summary["f_smoothness"] == signal_smoothness)
    ].copy()
    eps_values = sorted(plot_df["eps_smoothness"].dropna().unique())
    if not eps_values:
        raise ValueError(f"No rejection-probability summaries for N={sample_size}, s_m={signal_smoothness}.")

    fig, axes = plt.subplots(
        1,
        len(eps_values),
        figsize=(SINGLE_ROW_PANEL_WIDTH * len(eps_values), SINGLE_ROW_PANEL_HEIGHT),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes[0]

    for col_idx, eps_value in enumerate(eps_values):
        ax = axes[col_idx]
        panel = plot_df.loc[plot_df["eps_smoothness"] == eps_value].copy()
        _plot_rejection_lines(ax, panel, show_legend=False)
        ax.set_title(
            r"$s_\epsilon$ = " + _format_smoothness(eps_value),
            pad=GRID_COLUMN_TITLE_PAD,
            fontsize=FACET_LABEL_SIZE,
            fontweight="semibold",
        )
        apply_axis_visibility(ax, include_x_axis=include_x_axis, include_y_axis=include_y_axis and col_idx == 0)
        ax.tick_params(axis="x", which="both", bottom=include_x_axis, labelbottom=include_x_axis)
        ax.tick_params(axis="y", which="both", left=include_y_axis and col_idx == 0, labelleft=include_y_axis and col_idx == 0)

    handles, labels = axes[0].get_legend_handles_labels()
    if include_legend and handles:
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, SINGLE_ROW_LEGEND_Y),
            fontsize=8.4,
            ncol=len(PROCEDURE_ORDER),
            title=None,
            columnspacing=1.0,
            handlelength=1.6,
            handletextpad=0.35,
        )
    else:
        remove_legends(fig)

    fig.suptitle(
        r"Rejection probability at 5% level ($s_m$ = "
        + _format_smoothness(signal_smoothness)
        + r", $N$ = "
        + str(int(sample_size))
        + ")",
        y=SINGLE_ROW_TITLE_Y,
        fontsize=12.0,
        fontweight="semibold",
    )
    if include_x_axis:
        fig.supxlabel(r"dependence strength $\rho$", y=SINGLE_ROW_XLABEL_Y, fontsize=AXIS_LABEL_SIZE)
    if include_y_axis:
        fig.supylabel("rejection probability", x=SINGLE_ROW_YLABEL_X, fontsize=AXIS_LABEL_SIZE)

    fig.subplots_adjust(
        left=SINGLE_ROW_LEFT if include_y_axis else 0.035,
        right=SINGLE_ROW_RIGHT,
        bottom=SINGLE_ROW_BOTTOM if include_x_axis else 0.08,
        top=SINGLE_ROW_TOP,
        wspace=SINGLE_ROW_WSPACE,
    )
    return fig


def plot_rejection_probability_panel(
    summary: pd.DataFrame,
    *,
    f_smoothness: float,
    eps_smoothness: float,
    sample_size: int = SAMPLE_SIZE,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
):
    panel = summary.loc[
        (summary["N"] == sample_size)
        & (summary["f_smoothness"] == f_smoothness)
        & (summary["eps_smoothness"] == eps_smoothness)
    ].copy()

    fig, ax = plt.subplots(figsize=(3.05, 2.3))
    _plot_rejection_lines(ax, panel, show_legend=include_legend)
    ax.set_title(
        r"$s_m$ = "
        + _format_smoothness(f_smoothness)
        + r", $s_\epsilon$ = "
        + _format_smoothness(eps_smoothness),
        pad=3,
        fontsize=PANEL_TITLE_SIZE,
        fontweight="semibold",
    )
    ax.set_xlabel(r"dependence strength $\rho$" if include_x_axis else "", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("rejection probability" if include_y_axis else "", fontsize=AXIS_LABEL_SIZE)
    apply_axis_visibility(ax, include_x_axis=include_x_axis, include_y_axis=include_y_axis)
    if not include_legend:
        remove_legends(fig)
    fig.tight_layout()
    return fig


preview_fig = plot_rejection_probability_grid(rejection_summary, signal_smoothness=SIGNAL_SMOOTHNESS, sample_sizes=GRID_SAMPLE_SIZES)
plt.show()

single_row_preview_fig = plot_rejection_probability_single_row(
    rejection_summary,
    sample_size=SINGLE_ROW_SAMPLE_SIZE,
    signal_smoothness=SIGNAL_SMOOTHNESS,
)
plt.show()


In [ ]:
saved_paths = []

if SAVE_FIGURES and SAVE_FULL_FIGURE:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for signal_smoothness in SIGNAL_SMOOTHNESS_VALUES:
        fig = plot_rejection_probability_grid(
            rejection_summary,
            signal_smoothness=signal_smoothness,
            sample_sizes=GRID_SAMPLE_SIZES,
            include_legend=True,
            include_x_axis=True,
            include_y_axis=True,
        )
        stem = (
            f"rejection_probability_m_smoothness_{slugify(_format_smoothness(signal_smoothness))}"
            f"_N_{'_'.join(map(str, GRID_SAMPLE_SIZES))}_full_grid"
        )
        for fmt in SAVE_FORMATS:
            path = FIGURE_DIR / f"{stem}.{fmt}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            saved_paths.append(path)
        plt.close(fig)

if SAVE_FIGURES and SAVE_SINGLE_ROW_FIGURE:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for signal_smoothness in SIGNAL_SMOOTHNESS_VALUES:
        for sample_size in SINGLE_ROW_SAMPLE_SIZES:
            fig = plot_rejection_probability_single_row(
                rejection_summary,
                sample_size=sample_size,
                signal_smoothness=signal_smoothness,
                include_legend=True,
                include_x_axis=True,
                include_y_axis=True,
            )
            stem = (
                f"rejection_probability_m_smoothness_{slugify(_format_smoothness(signal_smoothness))}"
                f"_N_{sample_size}_single_row"
            )
            for fmt in SAVE_FORMATS:
                path = FIGURE_DIR / f"{stem}.{fmt}"
                fig.savefig(path, dpi=300, bbox_inches="tight")
                saved_paths.append(path)
            plt.close(fig)

print(f"Saved {len(saved_paths)} figure files to {FIGURE_DIR}.")
if saved_paths:
    display(pd.DataFrame({"path": [str(path) for path in saved_paths]}))


## Reverse-Triangle Coverage

Empirical coverage of the 95% debiased reverse-triangle confidence sets, summarized as a table because the corresponding coverage plot is visually uninformative. This uses the same confidence-set summary machinery as `multivariate_sweep_results_summary.ipynb`.


In [ ]:
COVERAGE_METHOD = "bootstrap_debiased"
COVERAGE_METHOD_LABEL = "bootstrap debiased"
TABLE_DIR = OUTPUT / "tables/synthetic"

coverage_ci = build_ci_frame(favourability)
if not coverage_ci.empty:
    coverage_ci = coverage_ci.loc[
        (coverage_ci["method"].astype(str) == COVERAGE_METHOD)
        & (coverage_ci["rho"] >= RHO_MIN)
        & (coverage_ci["rho"] <= RHO_MAX)
    ].copy()

coverage_group_cols = ["method_label", "N", "rho", "f_smoothness", "eps_smoothness"]
coverage_summary = summarize_binary(coverage_ci, coverage_group_cols, "coverage", "coverage")
if not coverage_summary.empty:
    coverage_summary = coverage_summary.sort_values(["f_smoothness", "N", "eps_smoothness", "rho", "method_label"]).reset_index(drop=True)

coverage_table_source = coverage_summary.loc[
    (coverage_summary["method_label"].astype(str) == COVERAGE_METHOD_LABEL)
    & (coverage_summary["f_smoothness"] == SIGNAL_SMOOTHNESS)
    & (coverage_summary["N"].isin(GRID_SAMPLE_SIZES))
].copy()


coverage_table_long = (
    coverage_table_source.groupby(["N", "eps_smoothness"], dropna=False, observed=True)
    .agg(
        rho_min=("rho", "min"),
        rho_max=("rho", "max"),
        coverage_min=("coverage", "min"),
        coverage_mean=("coverage", "mean"),
        coverage_max=("coverage", "max"),
        min_runs_per_rho=("runs", "min"),
    )
    .reset_index()
)
coverage_table_long["coverage_range"] = coverage_table_long.apply(
    lambda row: f"{row['coverage_min']:.2f}" if np.isclose(row["coverage_min"], row["coverage_max"]) else f"{row['coverage_min']:.2f}-{row['coverage_max']:.2f}",
    axis=1,
)

coverage_table = coverage_table_long.pivot(index="N", columns="eps_smoothness", values="coverage_range")
coverage_table = coverage_table.reindex(index=GRID_SAMPLE_SIZES)
coverage_table = coverage_table.reindex(sorted(coverage_table.columns), axis=1)
coverage_table.columns = [r"$s_\epsilon$=" + _format_smoothness(value) for value in coverage_table.columns]
coverage_table.index.name = r"$N$"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
coverage_table_stem = (
    f"coverage_probability_reverse_triangle_debiased_m_smoothness_{slugify(_format_smoothness(SIGNAL_SMOOTHNESS))}"
    f"_N_{'_'.join(map(str, GRID_SAMPLE_SIZES))}_table"
)
coverage_table_csv = TABLE_DIR / f"{coverage_table_stem}.csv"
coverage_table_tex = TABLE_DIR / f"{coverage_table_stem}.tex"
coverage_table_long_csv = TABLE_DIR / f"{coverage_table_stem}_long.csv"

coverage_table.to_csv(coverage_table_csv)
coverage_table_long.to_csv(coverage_table_long_csv, index=False)
coverage_table.to_latex(
    coverage_table_tex,
    escape=False,
    caption=(
        r"Empirical coverage of nominal 95\% debiased reverse-triangle confidence sets. "
        r"Entries report the range over $\rho \in [0, 0.5]$."
    ),
    label="tab:synthetic_reverse_triangle_coverage",
)

print(f"Built {len(coverage_ci):,} debiased reverse-triangle CI rows and {len(coverage_summary):,} summarized coverage probabilities.")
print(f"Saved coverage table to {coverage_table_csv} and {coverage_table_tex}.")
display(coverage_table)


## Coverage Error-Bar Tables

CSV exports for per-rho empirical coverage with binomial 95% error bars. Delta-method rows at `rho=0` are retained as `NA` because the delta-method confidence set is not used at the null.


In [ ]:
TABLE_DIR = OUTPUT / "tables/synthetic"
COVERAGE_ERROR_BAR_DIR = TABLE_DIR / "coverage_error_bars"

COVERAGE_ERROR_BAR_SAMPLE_SIZES = GRID_SAMPLE_SIZES
COVERAGE_ERROR_BAR_SIGNAL_SMOOTHNESS_VALUES = SIGNAL_SMOOTHNESS_VALUES
COVERAGE_ERROR_BAR_NOISE_SMOOTHNESS_VALUES = [0.25, 0.5, 0.75]
COVERAGE_ERROR_BAR_RHOS = [round(float(rho), 10) for rho in np.linspace(RHO_MIN, RHO_MAX, 6)]

REVERSE_TRIANGLE_COVERAGE_METHODS = ["bootstrap_debiased"]
DELTA_COVERAGE_METHODS = [method for method in METHOD_ORDER if method.startswith("delta_")]
COVERAGE_ERROR_BAR_SAMPLE_SIZE_STEM = "_".join(map(str, COVERAGE_ERROR_BAR_SAMPLE_SIZES))

if "_format_smoothness" not in globals():
    def _format_smoothness(value: float) -> str:
        return f"{float(value):g}"


def _format_coverage_error_bar(row: pd.Series) -> str:
    coverage = row["coverage"]
    if pd.isna(coverage):
        return "NA"
    return f"{coverage:.3f} (-{row['coverage_err_low']:.3f}, +{row['coverage_err_high']:.3f})"


def _coverage_method_label_map(ci: pd.DataFrame, methods: list[str]) -> dict[str, str]:
    labels: dict[str, str] = {}
    if not ci.empty and {"method", "method_label"}.issubset(ci.columns):
        method_block = ci.loc[ci["method"].astype(str).isin(methods), ["method", "method_label"]].drop_duplicates()
        for _, row in method_block.iterrows():
            labels[str(row["method"])] = str(row["method_label"])
    for method in methods:
        labels.setdefault(method, method.replace("_", " "))
    return labels


def _normalise_coverage_ci_subset(ci: pd.DataFrame, methods: list[str]) -> pd.DataFrame:
    if ci.empty:
        return ci.copy()
    subset = ci.copy()
    subset["method"] = subset["method"].astype(str)
    subset["method_label"] = subset["method_label"].astype(str)
    for col in ["N", "rho", "f_smoothness", "eps_smoothness"]:
        subset[col] = pd.to_numeric(subset[col], errors="coerce")
    subset["rho"] = subset["rho"].round(10)
    subset["f_smoothness"] = subset["f_smoothness"].round(10)
    subset["eps_smoothness"] = subset["eps_smoothness"].round(10)
    subset = subset.loc[
        subset["method"].isin(methods)
        & subset["N"].isin(COVERAGE_ERROR_BAR_SAMPLE_SIZES)
        & subset["rho"].isin(COVERAGE_ERROR_BAR_RHOS)
        & subset["f_smoothness"].isin(COVERAGE_ERROR_BAR_SIGNAL_SMOOTHNESS_VALUES)
        & subset["eps_smoothness"].isin(COVERAGE_ERROR_BAR_NOISE_SMOOTHNESS_VALUES)
    ].copy()
    if not subset.empty:
        subset["N"] = subset["N"].astype(int)
    return subset


def build_coverage_error_bar_summary(
    ci: pd.DataFrame,
    methods: list[str],
    *,
    delta_na_at_zero: bool = False,
) -> pd.DataFrame:
    methods = [str(method) for method in methods]
    labels = _coverage_method_label_map(ci, methods)
    selected = _normalise_coverage_ci_subset(ci, methods)
    group_cols = ["method", "method_label", "N", "rho", "f_smoothness", "eps_smoothness"]
    value_cols = ["runs", "coverage", "coverage_lower", "coverage_upper", "coverage_err_low", "coverage_err_high"]

    if selected.empty:
        summary = pd.DataFrame(columns=group_cols + value_cols)
    else:
        summary = summarize_binary(selected, group_cols, "coverage", "coverage")
        summary["method"] = summary["method"].astype(str)
        summary["method_label"] = summary["method_label"].astype(str)
        for col in ["rho", "f_smoothness", "eps_smoothness"]:
            summary[col] = pd.to_numeric(summary[col], errors="coerce").round(10)
        summary["N"] = pd.to_numeric(summary["N"], errors="coerce").astype("Int64")

    grid = pd.MultiIndex.from_product(
        [
            methods,
            COVERAGE_ERROR_BAR_SAMPLE_SIZES,
            COVERAGE_ERROR_BAR_RHOS,
            COVERAGE_ERROR_BAR_SIGNAL_SMOOTHNESS_VALUES,
            COVERAGE_ERROR_BAR_NOISE_SMOOTHNESS_VALUES,
        ],
        names=["method", "N", "rho", "f_smoothness", "eps_smoothness"],
    ).to_frame(index=False)
    grid["method_label"] = grid["method"].map(labels)
    grid = grid[group_cols]

    summary = grid.merge(summary, on=group_cols, how="left")
    if delta_na_at_zero:
        zero_rho = np.isclose(summary["rho"].astype(float), 0.0)
        summary.loc[zero_rho, value_cols] = np.nan

    summary["coverage_error_bar"] = summary.apply(_format_coverage_error_bar, axis=1)
    return summary[
        group_cols
        + value_cols
        + ["coverage_error_bar"]
    ].sort_values(["f_smoothness", "eps_smoothness", "rho", "N", "method"]).reset_index(drop=True)


def _coverage_table_filename(prefix: str, f_smoothness: float, eps_smoothness: float) -> Path:
    return COVERAGE_ERROR_BAR_DIR / (
        f"{prefix}_m_smoothness_{slugify(_format_smoothness(f_smoothness))}"
        f"_eps_smoothness_{slugify(_format_smoothness(eps_smoothness))}"
        f"_N_{COVERAGE_ERROR_BAR_SAMPLE_SIZE_STEM}_table.csv"
    )


def _write_smoothness_specific_coverage_tables(
    reverse_summary: pd.DataFrame,
    delta_summary: pd.DataFrame,
) -> list[Path]:
    saved: list[Path] = []
    reverse_columns = [f"rho_{slugify(_format_smoothness(rho))}" for rho in COVERAGE_ERROR_BAR_RHOS]
    for f_smoothness in COVERAGE_ERROR_BAR_SIGNAL_SMOOTHNESS_VALUES:
        for eps_smoothness in COVERAGE_ERROR_BAR_NOISE_SMOOTHNESS_VALUES:
            reverse_panel = reverse_summary.loc[
                (reverse_summary["f_smoothness"] == f_smoothness)
                & (reverse_summary["eps_smoothness"] == eps_smoothness)
            ]
            reverse_table = reverse_panel.pivot_table(
                index="N",
                columns="rho",
                values="coverage_error_bar",
                aggfunc="first",
            ).reindex(index=COVERAGE_ERROR_BAR_SAMPLE_SIZES, columns=COVERAGE_ERROR_BAR_RHOS)
            reverse_table.columns = reverse_columns
            reverse_table.index.name = "N"
            reverse_path = _coverage_table_filename("coverage_error_bars_reverse_triangle", f_smoothness, eps_smoothness)
            reverse_table.to_csv(reverse_path)
            saved.append(reverse_path)

            delta_panel = delta_summary.loc[
                (delta_summary["f_smoothness"] == f_smoothness)
                & (delta_summary["eps_smoothness"] == eps_smoothness)
            ]
            delta_index = pd.MultiIndex.from_product(
                [COVERAGE_ERROR_BAR_RHOS, COVERAGE_ERROR_BAR_SAMPLE_SIZES],
                names=["rho", "N"],
            )
            delta_table = delta_panel.pivot_table(
                index=["rho", "N"],
                columns="method",
                values="coverage_error_bar",
                aggfunc="first",
            ).reindex(index=delta_index, columns=DELTA_COVERAGE_METHODS)
            delta_path = _coverage_table_filename("coverage_error_bars_delta", f_smoothness, eps_smoothness)
            delta_table.to_csv(delta_path)
            saved.append(delta_path)
    return saved


coverage_all_ci = build_ci_frame(favourability)
reverse_coverage_error_bars = build_coverage_error_bar_summary(
    coverage_all_ci,
    REVERSE_TRIANGLE_COVERAGE_METHODS,
)
reverse_coverage_error_bars.insert(0, "confidence_set", "reverse_triangle_debiased")
delta_coverage_error_bars = build_coverage_error_bar_summary(
    coverage_all_ci,
    DELTA_COVERAGE_METHODS,
    delta_na_at_zero=True,
)
delta_coverage_error_bars.insert(0, "confidence_set", "delta_method")

reverse_coverage_error_bar_table = reverse_coverage_error_bars.pivot_table(
    index=["f_smoothness", "eps_smoothness", "N"],
    columns="rho",
    values="coverage_error_bar",
    aggfunc="first",
).reindex(columns=COVERAGE_ERROR_BAR_RHOS)
reverse_coverage_error_bar_table.columns = [f"rho_{slugify(_format_smoothness(rho))}" for rho in COVERAGE_ERROR_BAR_RHOS]

delta_coverage_error_bar_table = delta_coverage_error_bars.pivot_table(
    index=["f_smoothness", "eps_smoothness", "rho", "N"],
    columns="method",
    values="coverage_error_bar",
    aggfunc="first",
).reindex(columns=DELTA_COVERAGE_METHODS)

COVERAGE_ERROR_BAR_DIR.mkdir(parents=True, exist_ok=True)
coverage_error_bar_paths = [
    COVERAGE_ERROR_BAR_DIR / f"coverage_error_bars_reverse_triangle_N_{COVERAGE_ERROR_BAR_SAMPLE_SIZE_STEM}_long.csv",
    COVERAGE_ERROR_BAR_DIR / f"coverage_error_bars_delta_N_{COVERAGE_ERROR_BAR_SAMPLE_SIZE_STEM}_long.csv",
    COVERAGE_ERROR_BAR_DIR / f"coverage_error_bars_reverse_triangle_N_{COVERAGE_ERROR_BAR_SAMPLE_SIZE_STEM}_all_smoothness.csv",
    COVERAGE_ERROR_BAR_DIR / f"coverage_error_bars_delta_N_{COVERAGE_ERROR_BAR_SAMPLE_SIZE_STEM}_all_smoothness.csv",
]

reverse_coverage_error_bars.to_csv(coverage_error_bar_paths[0], index=False)
delta_coverage_error_bars.to_csv(coverage_error_bar_paths[1], index=False)
reverse_coverage_error_bar_table.to_csv(coverage_error_bar_paths[2])
delta_coverage_error_bar_table.to_csv(coverage_error_bar_paths[3])
coverage_error_bar_paths.extend(
    _write_smoothness_specific_coverage_tables(reverse_coverage_error_bars, delta_coverage_error_bars)
)

print(f"Saved {len(coverage_error_bar_paths)} coverage error-bar CSVs to {COVERAGE_ERROR_BAR_DIR}.")
display(pd.DataFrame({"path": [str(path) for path in coverage_error_bar_paths]}))
display(reverse_coverage_error_bar_table.head(9))
display(delta_coverage_error_bar_table.head(18))


## Split-Test Miscalibration

Rejection probability of the split-standard residual HSIC test across training split ratios, using the `better_permutation_calibration_*` runs.


In [ ]:
BETTER_PERMUTATION_PATTERN = os.environ.get("HSIC_SPLIT_PATTERN", "better_permutation_calibration_*")
BETTER_SPLIT_CACHE = BASE / ".better_permutation_standard_residual_split_tests.pkl"
BETTER_SPLIT_CACHE_VERSION = 1
SPLIT_SIGNAL_SMOOTHNESS = 1.0
SPLIT_RHOS = [0.0, 0.3]
SPLIT_N_VALUES = [250, 500, 1000, 1500]
SPLIT_SAVE_FIGURES = SAVE_FIGURES


def _results_fingerprint(paths: list[Path]) -> tuple[tuple[str, int, int], ...]:
    return tuple((str(path), path.stat().st_mtime_ns, path.stat().st_size) for path in paths)


def _load_split_cache(cache_path: Path, fingerprint):
    if not cache_path.exists():
        return None
    try:
        with cache_path.open("rb") as handle:
            cached = pickle.load(handle)
    except (OSError, EOFError, pickle.PickleError, AttributeError, ValueError):
        return None
    if cached.get("version") != BETTER_SPLIT_CACHE_VERSION:
        return None
    # Preserve campaign/job, mtime and size checks across checkout prefixes.
    def relative_fingerprint(entries):
        return sorted((tuple(Path(path).parts[-3:]), mtime, size) for path, mtime, size in entries)
    if relative_fingerprint(cached.get("fingerprint", ())) != relative_fingerprint(fingerprint):
        return None
    rows = cached.get("rows")
    if not isinstance(rows, list):
        return None
    return [dict(row, results_path=str(BASE / row["source_run_name"] / row["run_name"] / "results.yaml")) for row in rows]


def _write_split_cache(cache_path: Path, fingerprint, rows: list[dict]) -> None:
    payload = {"version": BETTER_SPLIT_CACHE_VERSION, "fingerprint": fingerprint, "rows": rows}
    try:
        with cache_path.open("wb") as handle:
            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    except OSError:
        pass


def load_better_permutation_split_tests(pattern: str = BETTER_PERMUTATION_PATTERN) -> pd.DataFrame:
    run_dirs = sorted(BASE.glob(pattern))
    yaml_paths = sorted(path for run_dir in run_dirs for path in run_dir.glob("*/results.yaml"))
    fingerprint = _results_fingerprint(yaml_paths)
    cached_rows = _load_split_cache(BETTER_SPLIT_CACHE, fingerprint)
    if cached_rows is None:
        rows = []
        for yaml_path in yaml_paths:
            raw = yaml.safe_load(yaml_path.read_text()) or {}
            cfg = raw.get("config", {}) if isinstance(raw, dict) else {}
            entries = raw.get("standard_residual_hsic", []) if isinstance(raw, dict) else []
            if not isinstance(entries, list):
                continue
            num_folds = cfg.get("num_folds") or 5
            for entry in entries:
                if not isinstance(entry, dict):
                    continue
                p_value = entry.get("naive_hsic_u_p_value")
                fit_prefix_folds = entry.get("fit_prefix_folds")
                if p_value is None or fit_prefix_folds is None:
                    continue
                train_ratio = float(fit_prefix_folds) / float(num_folds)
                rows.append(
                    {
                        "source_run_name": yaml_path.parents[1].name,
                        "run_name": yaml_path.parent.name,
                        "results_path": str(yaml_path),
                        "N": cfg.get("N"),
                        "rho": cfg.get("rho"),
                        "f_smoothness": cfg.get("f_smoothness"),
                        "eps_smoothness": cfg.get("eps_smoothness"),
                        "random_key": cfg.get("random_key"),
                        "num_folds": num_folds,
                        "fit_prefix_folds": fit_prefix_folds,
                        "train_ratio": train_ratio,
                        "train_percent": 100.0 * train_ratio,
                        "n_fit": entry.get("n_fit"),
                        "n_test": entry.get("n_test"),
                        "p_value": p_value,
                    }
                )
        # Input archives are read-only; no cache is written here.
    else:
        rows = cached_rows

    df = pd.DataFrame(rows)
    for col in ["N", "rho", "f_smoothness", "eps_smoothness", "random_key", "fit_prefix_folds", "train_ratio", "train_percent", "n_fit", "n_test", "p_value"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if not df.empty:
        dedupe_cols = ["N", "rho", "f_smoothness", "eps_smoothness", "random_key", "fit_prefix_folds"]
        df = (
            df.sort_values(["source_run_name", "run_name"])
            .drop_duplicates(dedupe_cols, keep="last")
            .reset_index(drop=True)
        )
        df["reject"] = df["p_value"] < ALPHA
    return df


def summarize_split_rejection(tests: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["N", "rho", "f_smoothness", "eps_smoothness", "fit_prefix_folds", "train_percent"]
    for key, block in tests.groupby(group_cols, dropna=False, observed=True):
        successes = int(block["reject"].sum())
        total = len(block)
        reject_rate, lower, upper = wilson_interval(successes, total)
        row = dict(zip(group_cols, key))
        row.update(
            {
                "runs": total,
                "rejects": successes,
                "reject_rate": reject_rate,
                "reject_rate_lower": lower,
                "reject_rate_upper": upper,
                "reject_rate_err_low": reject_rate - lower,
                "reject_rate_err_high": upper - reject_rate,
            }
        )
        rows.append(row)
    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary = summary.sort_values(["rho", "f_smoothness", "N", "train_percent"]).reset_index(drop=True)
    return summary


split_tests = load_better_permutation_split_tests()
split_summary = summarize_split_rejection(split_tests)
print(f"Loaded {len(split_tests):,} split-test rows and built {len(split_summary):,} summaries.")
display(split_summary.round(4))

if split_tests.empty:
    raise FileNotFoundError(f"No split-test results under {BASE}")
split_tests.to_csv(TABLE_DIR / "split_input_rows.csv.gz", index=False)
split_summary.to_csv(TABLE_DIR / "split_rejection_summary.csv", index=False)

In [ ]:
# Manual layout knobs for split-test miscalibration figures.
SPLIT_PANEL_WIDTH = 3.8  # Width, in inches, of each rho-specific split-test figure.
SPLIT_PANEL_HEIGHT = 2.7  # Height, in inches, of each rho-specific split-test figure.
SPLIT_LEFT = 0.18  # Left edge of plotting area; higher moves axes right.
SPLIT_RIGHT = 0.98  # Right edge of plotting area.
SPLIT_BOTTOM = 0.16  # Bottom edge of plotting area; higher creates more room for x label.
SPLIT_TOP = 0.86  # Top edge of plotting area; lower creates more room for title/legend.
SPLIT_TITLE_Y = 0.97  # Vertical position of title.
SPLIT_LEGEND_Y = 0.88  # Vertical position of legend.
SPLIT_AXIS_LABEL_SIZE = 9.0  # Font size for axis labels.
SPLIT_TITLE_SIZE = 10.5  # Font size for title.
SPLIT_LEGEND_FONT_SIZE = 8.2  # Font size for legend labels.

SPLIT_COMBINED_PANEL_WIDTH = 2.65  # Width, in inches, of each rho panel in the combined figure.
SPLIT_COMBINED_PANEL_HEIGHT = 2.55  # Height, in inches, of the combined split-test figure.
SPLIT_COMBINED_LEFT = 0.12  # Left edge of combined subplot area.
SPLIT_COMBINED_RIGHT = 0.99  # Right edge of combined subplot area.
SPLIT_COMBINED_BOTTOM = 0.19  # Bottom edge of combined subplot area.
SPLIT_COMBINED_TOP = 0.84  # Top edge of combined subplot area.
SPLIT_COMBINED_WSPACE = 0.12  # Horizontal spacing between rho panels.
SPLIT_COMBINED_TITLE_Y = 0.98  # Vertical position of combined figure title.
SPLIT_COMBINED_LEGEND_LOC = "upper right"  # Legend location inside the left rho panel.


def _split_n_order(df: pd.DataFrame, n_values: list[int] | tuple[int, ...] | None = None) -> list[int]:
    available = [int(value) for value in sorted(df["N"].dropna().unique())]
    if n_values is None:
        return available
    return [int(value) for value in n_values if int(value) in available]


def plot_split_miscalibration(
    summary: pd.DataFrame,
    *,
    rho: float,
    signal_smoothness: float = SPLIT_SIGNAL_SMOOTHNESS,
    n_values: list[int] | tuple[int, ...] | None = SPLIT_N_VALUES,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
):
    plot_df = summary.loc[
        (summary["rho"] == rho)
        & (summary["f_smoothness"] == signal_smoothness)
    ].copy()
    if plot_df.empty:
        raise ValueError(f"No split-test summaries for rho={rho}, s_m={signal_smoothness}.")

    n_values = _split_n_order(plot_df, n_values=n_values)
    colors = plt.get_cmap("viridis")(np.linspace(0.10, 0.90, len(n_values)))
    n_palette = dict(zip(n_values, colors))

    fig, ax = plt.subplots(figsize=(SPLIT_PANEL_WIDTH, SPLIT_PANEL_HEIGHT))
    for n_value in n_values:
        block = plot_df.loc[plot_df["N"] == n_value].sort_values("train_percent")
        if block.empty:
            continue
        yerr = np.vstack([block["reject_rate_err_low"].to_numpy(), block["reject_rate_err_high"].to_numpy()])
        ax.errorbar(
            block["train_percent"],
            block["reject_rate"],
            yerr=yerr,
            label=f"N={n_value}",
            color=n_palette[n_value],
            marker="o",
            linestyle="-",
            linewidth=1.4,
            markersize=4.0,
            capsize=2.5,
            capthick=0.9,
            elinewidth=0.9,
        )

    ax.axhline(ALPHA, color="0.25", linestyle=":", linewidth=1.0, zorder=0)
    ax.set_ylim(-0.025, 1.025)
    ax.set_xticks([20, 40, 60, 80])
    ax.set_yticks(np.linspace(0.0, 1.0, 6))
    ax.grid(True, axis="y", color="0.88", linewidth=0.8)
    ax.grid(False, axis="x")

    ax.set_xlabel("training split (%)" if include_x_axis else "", fontsize=SPLIT_AXIS_LABEL_SIZE)
    ax.set_ylabel("rejection probability" if include_y_axis else "", fontsize=SPLIT_AXIS_LABEL_SIZE)
    apply_axis_visibility(ax, include_x_axis=include_x_axis, include_y_axis=include_y_axis)

    if include_legend:
        ax.legend(
            title=None,
            loc="upper center",
            bbox_to_anchor=(0.5, SPLIT_LEGEND_Y),
            ncol=2,
            fontsize=SPLIT_LEGEND_FONT_SIZE,
        )
    else:
        remove_legends(fig)

    fig.suptitle(
        r"Split-test rejection probability ($s_m$ = " + _format_smoothness(signal_smoothness) + rf", $\rho$ = {rho:g})",
        y=SPLIT_TITLE_Y,
        fontsize=SPLIT_TITLE_SIZE,
        fontweight="semibold",
    )
    fig.subplots_adjust(left=SPLIT_LEFT, right=SPLIT_RIGHT, bottom=SPLIT_BOTTOM, top=SPLIT_TOP)
    return fig


def _draw_split_miscalibration_panel(
    ax,
    plot_df: pd.DataFrame,
    *,
    n_values: list[int] | tuple[int, ...] | None = SPLIT_N_VALUES,
    include_x_axis: bool,
    include_y_axis: bool,
) -> None:
    n_values = _split_n_order(plot_df, n_values=n_values)
    colors = plt.get_cmap("viridis")(np.linspace(0.10, 0.90, len(n_values)))
    n_palette = dict(zip(n_values, colors))
    for n_value in n_values:
        block = plot_df.loc[plot_df["N"] == n_value].sort_values("train_percent")
        if block.empty:
            continue
        yerr = np.vstack([block["reject_rate_err_low"].to_numpy(), block["reject_rate_err_high"].to_numpy()])
        ax.errorbar(
            block["train_percent"],
            block["reject_rate"],
            yerr=yerr,
            label=f"N={n_value}",
            color=n_palette[n_value],
            marker="o",
            linestyle="-",
            linewidth=1.2,
            markersize=3.5,
            capsize=2.0,
            capthick=0.8,
            elinewidth=0.8,
        )
    ax.axhline(ALPHA, color="0.25", linestyle=":", linewidth=1.0, zorder=0)
    ax.set_ylim(-0.025, 1.025)
    ax.set_xticks([20, 40, 60, 80])
    ax.set_yticks(np.linspace(0.0, 1.0, 6))
    ax.grid(True, axis="y", color="0.88", linewidth=0.8)
    ax.grid(False, axis="x")
    ax.set_xlabel("training split (%)" if include_x_axis else "", fontsize=SPLIT_AXIS_LABEL_SIZE)
    ax.set_ylabel("rejection probability" if include_y_axis else "", fontsize=SPLIT_AXIS_LABEL_SIZE)
    apply_axis_visibility(ax, include_x_axis=include_x_axis, include_y_axis=include_y_axis)


def plot_split_miscalibration_combined(
    summary: pd.DataFrame,
    *,
    rhos: list[float] | tuple[float, ...] = SPLIT_RHOS,
    signal_smoothness: float = SPLIT_SIGNAL_SMOOTHNESS,
    n_values: list[int] | tuple[int, ...] | None = SPLIT_N_VALUES,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
):
    fig, axes = plt.subplots(
        1,
        len(rhos),
        figsize=(SPLIT_COMBINED_PANEL_WIDTH * len(rhos), SPLIT_COMBINED_PANEL_HEIGHT),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes[0]
    for col_idx, rho_value in enumerate(rhos):
        ax = axes[col_idx]
        plot_df = summary.loc[
            (summary["rho"] == rho_value)
            & (summary["f_smoothness"] == signal_smoothness)
        ].copy()
        if plot_df.empty:
            raise ValueError(f"No split-test summaries for rho={rho_value}, s_m={signal_smoothness}.")
        _draw_split_miscalibration_panel(
            ax,
            plot_df,
            n_values=n_values,
            include_x_axis=include_x_axis,
            include_y_axis=include_y_axis and col_idx == 0,
        )
        ax.set_title(r"$\rho$ = " + _format_smoothness(rho_value), pad=3, fontsize=FACET_LABEL_SIZE, fontweight="semibold")
        if col_idx > 0:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)

    if include_legend:
        axes[0].legend(
            title=None,
            loc=SPLIT_COMBINED_LEGEND_LOC,
            fontsize=SPLIT_LEGEND_FONT_SIZE,
            handlelength=1.4,
            borderaxespad=0.3,
            labelspacing=0.35,
            handletextpad=0.35,
        )
    else:
        remove_legends(fig)

    fig.suptitle(
        r"Split-test rejection probability at 5% level ($s_m$ = " + _format_smoothness(signal_smoothness) + ")",
        y=SPLIT_COMBINED_TITLE_Y,
        fontsize=SPLIT_TITLE_SIZE,
        fontweight="semibold",
    )
    fig.subplots_adjust(
        left=SPLIT_COMBINED_LEFT,
        right=SPLIT_COMBINED_RIGHT,
        bottom=SPLIT_COMBINED_BOTTOM,
        top=SPLIT_COMBINED_TOP,
        wspace=SPLIT_COMBINED_WSPACE,
    )
    return fig


split_preview_fig = plot_split_miscalibration_combined(
    split_summary,
    rhos=SPLIT_RHOS,
    signal_smoothness=SPLIT_SIGNAL_SMOOTHNESS,
    n_values=SPLIT_N_VALUES,
)
plt.show()


In [ ]:
# Narrow vertical variant of the split-test rejection probability figure.
SPLIT_VERTICAL_PANEL_WIDTH = 2  # Width, in inches, of the one-column vertical figure.
SPLIT_VERTICAL_PANEL_HEIGHT = 2  # Height, in inches, allocated to each rho panel.
SPLIT_VERTICAL_LEFT = 0.2  # Left edge of vertical subplot area.
SPLIT_VERTICAL_RIGHT = 0.98  # Right edge of vertical subplot area.
SPLIT_VERTICAL_BOTTOM = 0.11  # Bottom edge of vertical subplot area.
SPLIT_VERTICAL_TOP = 0.90  # Top edge of vertical subplot area.
SPLIT_VERTICAL_HSPACE = 0.2  # Vertical spacing between rho panels.
SPLIT_VERTICAL_TITLE_Y = 0.99  # Vertical position of vertical figure title.
SPLIT_VERTICAL_YLABEL_X = 0.0  # Horizontal position of the shared y label.
SPLIT_VERTICAL_LEGEND_LOC = "upper right"  # Legend location inside the top rho panel.


def plot_split_miscalibration_vertical(
    summary: pd.DataFrame,
    *,
    rhos: list[float] | tuple[float, ...] = SPLIT_RHOS,
    signal_smoothness: float = SPLIT_SIGNAL_SMOOTHNESS,
    n_values: list[int] | tuple[int, ...] | None = SPLIT_N_VALUES,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
):
    fig, axes = plt.subplots(
        len(rhos),
        1,
        figsize=(SPLIT_VERTICAL_PANEL_WIDTH, SPLIT_VERTICAL_PANEL_HEIGHT * len(rhos)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes[:, 0]
    for row_idx, rho_value in enumerate(rhos):
        ax = axes[row_idx]
        plot_df = summary.loc[
            (summary["rho"] == rho_value)
            & (summary["f_smoothness"] == signal_smoothness)
        ].copy()
        if plot_df.empty:
            raise ValueError(f"No split-test summaries for rho={rho_value}, s_m={signal_smoothness}.")
        _draw_split_miscalibration_panel(
            ax,
            plot_df,
            n_values=n_values,
            include_x_axis=include_x_axis and row_idx == len(rhos) - 1,
            include_y_axis=include_y_axis,
        )
        if include_y_axis:
            ax.set_ylabel("")
        ax.set_title(r"$\rho$ = " + _format_smoothness(rho_value), pad=3, fontsize=FACET_LABEL_SIZE, fontweight="semibold")

    if include_legend:
        axes[0].legend(
            title=None,
            loc=SPLIT_VERTICAL_LEGEND_LOC,
            ncol=2,
            fontsize=SPLIT_LEGEND_FONT_SIZE,
            handlelength=1.25,
            borderaxespad=0.3,
            columnspacing=0.7,
            labelspacing=0.3,
            handletextpad=0.35,
        )
    else:
        remove_legends(fig)

    fig.suptitle(
        r"$s_m$ = " + _format_smoothness(signal_smoothness),
        y=SPLIT_VERTICAL_TITLE_Y,
        fontsize=SPLIT_TITLE_SIZE,
        fontweight="semibold",
    )
    if include_y_axis:
        fig.supylabel("rejection probability", x=SPLIT_VERTICAL_YLABEL_X, fontsize=SPLIT_AXIS_LABEL_SIZE)
    fig.subplots_adjust(
        left=SPLIT_VERTICAL_LEFT,
        right=SPLIT_VERTICAL_RIGHT,
        bottom=SPLIT_VERTICAL_BOTTOM,
        top=SPLIT_VERTICAL_TOP,
        hspace=SPLIT_VERTICAL_HSPACE,
    )
    return fig


split_vertical_preview_fig = plot_split_miscalibration_vertical(
    split_summary,
    rhos=SPLIT_RHOS,
    signal_smoothness=SPLIT_SIGNAL_SMOOTHNESS,
    n_values=SPLIT_N_VALUES,
)
plt.show()

In [ ]:
split_saved_paths = []

if SPLIT_SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    base_stem = (
        f"split_test_rejection_m_smoothness_{slugify(_format_smoothness(SPLIT_SIGNAL_SMOOTHNESS))}"
        f"_rho_{'_'.join(slugify(_format_smoothness(rho)) for rho in SPLIT_RHOS)}_N_{'_'.join(map(str, SPLIT_N_VALUES))}"
    )
    for layout_name, plot_factory in [
        ("combined", plot_split_miscalibration_combined),
        ("vertical", plot_split_miscalibration_vertical),
    ]:
        fig = plot_factory(
            split_summary,
            rhos=SPLIT_RHOS,
            signal_smoothness=SPLIT_SIGNAL_SMOOTHNESS,
            n_values=SPLIT_N_VALUES,
            include_legend=True,
            include_x_axis=True,
            include_y_axis=True,
        )
        for fmt in SAVE_FORMATS:
            path = FIGURE_DIR / f"{base_stem}_{layout_name}.{fmt}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            split_saved_paths.append(path)
        plt.close(fig)

print(f"Saved {len(split_saved_paths)} split-test figure files to {FIGURE_DIR}.")
if split_saved_paths:
    display(pd.DataFrame({"path": [str(path) for path in split_saved_paths]}))


## Final QQ Calibration Comparison

QQ plots for one sample size, comparing the naive plug-in U-statistic calibration with the debiased U-statistic calibration across the selected rho values.


In [ ]:
# Manual layout knobs for final QQ calibration figures.
QQ_SAMPLE_SIZE = 500
QQ_SIGNAL_SMOOTHNESS = SIGNAL_SMOOTHNESS
QQ_EPS_SMOOTHNESS = 0.75
QQ_RHOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
QQ_MIN_RUNS = 20
QQ_SAVE_FIGURES = SAVE_FIGURES
QQ_SHOW_TITLE = True
QQ_DISCOVER_COMPLETE_SETTINGS = False
QQ_SAMPLE_SIZES = [250, 500, 1000, 1500]
QQ_SIGNAL_SMOOTHNESS_VALUES = [1.0, 1.5, 2.0]
QQ_EPS_SMOOTHNESS_VALUES = [0.25, 0.5, 0.75]

QQ_PANEL_WIDTH = 2.25
QQ_PANEL_HEIGHT = 2.05
QQ_LEFT = 0.08
QQ_RIGHT = 0.985
QQ_BOTTOM = 0.11
QQ_TOP_WITH_TITLE = 0.86
QQ_TOP_NO_TITLE = 0.94
QQ_WSPACE = 0.05
QQ_HSPACE = 0.15
QQ_TITLE_Y = 0.985
QQ_LEGEND_Y = 0.965
QQ_XLABEL_Y = 0.025
QQ_YLABEL_X = 0.015
QQ_AXIS_LABEL_SIZE = 9.5
QQ_TITLE_SIZE = 11.0
QQ_LEGEND_FONT_SIZE = 9.0
QQ_MARKER_SIZE = 7.0
QQ_MARKER_ALPHA = 0.82
QQ_LEGEND_MARKER_SIZE = 70.0
QQ_DIAGONAL_COLOR = "0.08"
QQ_DIAGONAL_LINEWIDTH = 1.6

QQ_ESTIMATOR_ORDER = ["plug-in U-statistic", "debiased U-statistic"]
QQ_ESTIMATOR_SPECS = [
    ("plug-in U-statistic", "estimators.naive_hsic_u"),
    ("debiased U-statistic", "estimators.debiased_hsic_u"),
]
QQ_ESTIMATOR_PALETTE = {
    "plug-in U-statistic": PROCEDURE_COLORS[0],
    "debiased U-statistic": PROCEDURE_COLORS[1],
}
QQ_ESTIMATOR_MARKERS = {"plug-in U-statistic": "o", "debiased U-statistic": "^"}


def discover_complete_qq_settings(
    df: pd.DataFrame,
    *,
    rhos: list[float] | tuple[float, ...] = QQ_RHOS,
    min_runs: int = QQ_MIN_RUNS,
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["N", "f_smoothness", "eps_smoothness", "min_runs_per_rho"])
    truth_col = "population_oracle.hsic"
    plot_df = df.copy()
    if truth_col not in plot_df.columns and "population_oracle.sqrt_hsic" in plot_df.columns:
        truth_col = "population_oracle.hsic_from_sqrt"
        plot_df[truth_col] = pd.to_numeric(plot_df["population_oracle.sqrt_hsic"], errors="coerce") ** 2
    required_cols = [truth_col, *[estimate_col for _, estimate_col in QQ_ESTIMATOR_SPECS]]
    missing_cols = [col for col in required_cols if col not in plot_df.columns]
    if missing_cols:
        raise ValueError("Need QQ columns to discover complete settings: " + ", ".join(missing_cols))

    rhos = [float(rho) for rho in rhos]
    key_cols = ["N", "f_smoothness", "eps_smoothness"]
    rows = []
    usable = plot_df.loc[plot_df["rho"].isin(rhos)].copy()
    for key, block in usable.groupby(key_cols, dropna=False, observed=True):
        min_counts = []
        complete = True
        for rho_value in rhos:
            rho_block = block.loc[block["rho"] == rho_value]
            for _, estimate_col in QQ_ESTIMATOR_SPECS:
                count = int(rho_block[[truth_col, estimate_col]].dropna().shape[0])
                min_counts.append(count)
                if count < min_runs:
                    complete = False
        if complete:
            rows.append(
                {
                    "N": int(key[0]),
                    "f_smoothness": float(key[1]),
                    "eps_smoothness": float(key[2]),
                    "min_runs_per_rho": min(min_counts),
                }
            )
    settings = pd.DataFrame(rows)
    if not settings.empty:
        settings = settings.sort_values(["N", "f_smoothness", "eps_smoothness"]).reset_index(drop=True)
    return settings


def build_final_qq_frame(
    df: pd.DataFrame,
    *,
    sample_size: int = QQ_SAMPLE_SIZE,
    rhos: list[float] | tuple[float, ...] = QQ_RHOS,
    signal_smoothness: float = QQ_SIGNAL_SMOOTHNESS,
    eps_smoothness: float | None = QQ_EPS_SMOOTHNESS,
    min_runs: int = QQ_MIN_RUNS,
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    truth_col = "population_oracle.hsic"
    if truth_col not in df.columns and "population_oracle.sqrt_hsic" in df.columns:
        df = df.copy()
        truth_col = "population_oracle.hsic_from_sqrt"
        df[truth_col] = pd.to_numeric(df["population_oracle.sqrt_hsic"], errors="coerce") ** 2
    if truth_col not in df.columns:
        raise ValueError("Need population_oracle.hsic or population_oracle.sqrt_hsic to build QQ calibration plots.")

    plot_df = df.loc[
        (df["N"] == sample_size)
        & (df["f_smoothness"] == signal_smoothness)
        & (df["rho"].isin(rhos))
    ].copy()
    if eps_smoothness is not None:
        plot_df = plot_df.loc[plot_df["eps_smoothness"] == eps_smoothness].copy()
    if plot_df.empty:
        return pd.DataFrame()

    id_cols = ["N", "rho", "f_smoothness", "eps_smoothness", "random_key", truth_col]
    frames = []
    for estimator_label, estimate_col in QQ_ESTIMATOR_SPECS:
        if estimate_col not in plot_df.columns:
            continue
        block = plot_df[id_cols].copy()
        block["estimator_label"] = estimator_label
        block["estimate"] = pd.to_numeric(plot_df[estimate_col], errors="coerce")
        block["truth"] = pd.to_numeric(plot_df[truth_col], errors="coerce")
        block = block.dropna(subset=["estimate", "truth"])
        if not block.empty:
            frames.append(block)
    if not frames:
        return pd.DataFrame()

    errors = pd.concat(frames, ignore_index=True)
    errors["centered_error"] = errors["estimate"] - errors["truth"]
    regime_cols = ["estimator_label", "N", "rho", "f_smoothness", "eps_smoothness"]
    errors["empirical_sd"] = errors.groupby(regime_cols, observed=True)["centered_error"].transform("std")
    errors["runs"] = errors.groupby(regime_cols, observed=True)["centered_error"].transform("size")
    errors = errors.loc[(errors["runs"] >= min_runs) & (errors["empirical_sd"] > 0.0)].copy()
    if errors.empty:
        return pd.DataFrame()
    errors["z"] = errors["centered_error"] / errors["empirical_sd"]

    normal = NormalDist()
    rows = []
    for (estimator_label, rho_value), block in errors.groupby(["estimator_label", "rho"], observed=True):
        z = np.sort(pd.to_numeric(block["z"], errors="coerce").dropna().to_numpy())
        if len(z) < min_runs:
            continue
        probs = (np.arange(1, len(z) + 1) - 0.5) / len(z)
        theoretical = np.array([normal.inv_cdf(float(prob)) for prob in probs])
        for normal_quantile, empirical_quantile in zip(theoretical, z):
            rows.append(
                {
                    "estimator_label": estimator_label,
                    "rho": rho_value,
                    "normal_quantile": normal_quantile,
                    "empirical_quantile": empirical_quantile,
                    "runs": len(z),
                }
            )
    qq_df = pd.DataFrame(rows)
    if not qq_df.empty:
        qq_df["estimator_label"] = pd.Categorical(qq_df["estimator_label"], categories=QQ_ESTIMATOR_ORDER, ordered=True)
        qq_df = qq_df.sort_values(["rho", "estimator_label", "normal_quantile"]).reset_index(drop=True)
    return qq_df


def plot_final_qq_grid(
    qq_df: pd.DataFrame,
    *,
    sample_size: int = QQ_SAMPLE_SIZE,
    rhos: list[float] | tuple[float, ...] = QQ_RHOS,
    signal_smoothness: float = QQ_SIGNAL_SMOOTHNESS,
    eps_smoothness: float | None = QQ_EPS_SMOOTHNESS,
    include_legend: bool = True,
    include_x_axis: bool = True,
    include_y_axis: bool = True,
    include_title: bool = QQ_SHOW_TITLE,
):
    if qq_df.empty:
        raise ValueError("No QQ rows available for the requested final calibration comparison.")
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(QQ_PANEL_WIDTH * 3, QQ_PANEL_HEIGHT * 2),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    qq_min = float(np.nanmin([qq_df["normal_quantile"].min(), qq_df["empirical_quantile"].min(), -3.0]))
    qq_max = float(np.nanmax([qq_df["normal_quantile"].max(), qq_df["empirical_quantile"].max(), 3.0]))
    pad = 0.05 * (qq_max - qq_min)
    qq_limits = (qq_min - pad, qq_max + pad)

    for idx, rho_value in enumerate(rhos):
        ax = axes.flat[idx]
        panel = qq_df.loc[qq_df["rho"] == rho_value]
        for estimator_label in QQ_ESTIMATOR_ORDER:
            block = panel.loc[panel["estimator_label"] == estimator_label]
            if block.empty:
                continue
            ax.scatter(
                block["normal_quantile"],
                block["empirical_quantile"],
                label=estimator_label,
                color=QQ_ESTIMATOR_PALETTE[estimator_label],
                marker=QQ_ESTIMATOR_MARKERS[estimator_label],
                s=QQ_MARKER_SIZE,
                alpha=QQ_MARKER_ALPHA,
                linewidths=0.0,
            )
        ax.plot(qq_limits, qq_limits, color=QQ_DIAGONAL_COLOR, linestyle="--", linewidth=QQ_DIAGONAL_LINEWIDTH, zorder=0)
        ax.set_title(r"$\rho$ = " + _format_smoothness(rho_value), pad=3, fontsize=FACET_LABEL_SIZE, fontweight="semibold")
        ax.set_xlim(qq_limits)
        ax.set_ylim(qq_limits)
        ax.set_xticks([-3, -2, -1, 0, 1, 2, 3])
        ax.set_yticks([-3, -2, -1, 0, 1, 2, 3])
        ax.grid(True, color="0.90", linewidth=0.8)
        row_idx, col_idx = divmod(idx, 3)
        ax.set_xlabel("")
        ax.set_ylabel("")
        apply_axis_visibility(ax, include_x_axis=include_x_axis and row_idx == 1, include_y_axis=include_y_axis and col_idx == 0)

    for ax in axes.flat[len(rhos):]:
        ax.axis("off")

    if include_legend:
        handles = [
            plt.Line2D(
                [],
                [],
                linestyle="",
                marker=QQ_ESTIMATOR_MARKERS[estimator_label],
                markersize=math.sqrt(QQ_LEGEND_MARKER_SIZE),
                markerfacecolor=QQ_ESTIMATOR_PALETTE[estimator_label],
                markeredgecolor=QQ_ESTIMATOR_PALETTE[estimator_label],
                alpha=1.0,
                label=estimator_label,
            )
            for estimator_label in QQ_ESTIMATOR_ORDER
            if not qq_df.loc[qq_df["estimator_label"] == estimator_label].empty
        ]
        labels = [handle.get_label() for handle in handles]
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, QQ_LEGEND_Y),
            ncol=len(labels),
            fontsize=QQ_LEGEND_FONT_SIZE,
            handlelength=1.2,
            columnspacing=1.2,
            handletextpad=0.35,
        )
    else:
        remove_legends(fig)

    if include_title:
        title = r"QQ normal calibration ($N$ = " + str(sample_size) + r", $s_m$ = " + _format_smoothness(signal_smoothness)
        if eps_smoothness is not None:
            title += r", $s_\epsilon$ = " + _format_smoothness(eps_smoothness)
        title += ")"
        fig.suptitle(title, y=QQ_TITLE_Y, fontsize=QQ_TITLE_SIZE, fontweight="semibold")
    if include_x_axis:
        fig.supxlabel("normal quantile", y=QQ_XLABEL_Y, fontsize=QQ_AXIS_LABEL_SIZE)
    if include_y_axis:
        fig.supylabel("empirical quantile", x=QQ_YLABEL_X, fontsize=QQ_AXIS_LABEL_SIZE)

    fig.subplots_adjust(
        left=QQ_LEFT,
        right=QQ_RIGHT,
        bottom=QQ_BOTTOM,
        top=QQ_TOP_WITH_TITLE if include_title else QQ_TOP_NO_TITLE,
        wspace=QQ_WSPACE,
        hspace=QQ_HSPACE,
    )
    return fig


final_qq_settings = pd.DataFrame(
    [
        {"N": sample_size, "f_smoothness": signal_smoothness, "eps_smoothness": eps_smoothness}
        for sample_size in QQ_SAMPLE_SIZES
        for signal_smoothness in QQ_SIGNAL_SMOOTHNESS_VALUES
        for eps_smoothness in QQ_EPS_SMOOTHNESS_VALUES
    ]
)
print(f"Configured {len(final_qq_settings):,} QQ settings across rho={QQ_RHOS}.")
display(final_qq_settings)

final_qq = build_final_qq_frame(
    favourability,
    sample_size=QQ_SAMPLE_SIZE,
    rhos=QQ_RHOS,
    signal_smoothness=QQ_SIGNAL_SMOOTHNESS,
    eps_smoothness=QQ_EPS_SMOOTHNESS,
)
print(
    f"Built {len(final_qq):,} preview QQ rows for "
    f"N={QQ_SAMPLE_SIZE}, "
    f"s_m={_format_smoothness(QQ_SIGNAL_SMOOTHNESS)}, "
    f"s_epsilon={_format_smoothness(QQ_EPS_SMOOTHNESS)}."
)
if not final_qq.empty:
    display(final_qq.groupby(["rho", "estimator_label"], observed=True)["runs"].first().reset_index())
    final_qq_preview_fig = plot_final_qq_grid(final_qq)
    plt.show()
else:
    available = favourability[["N", "rho", "f_smoothness", "eps_smoothness"]].drop_duplicates().sort_values(["N", "f_smoothness", "eps_smoothness", "rho"])
    display(available)


In [ ]:
final_qq_saved_paths = []

if QQ_SAVE_FIGURES and not final_qq_settings.empty:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for setting in final_qq_settings.itertuples(index=False):
        sample_size = int(setting.N)
        signal_smoothness = float(setting.f_smoothness)
        eps_smoothness = float(setting.eps_smoothness)
        qq_df = build_final_qq_frame(
            favourability,
            sample_size=sample_size,
            rhos=QQ_RHOS,
            signal_smoothness=signal_smoothness,
            eps_smoothness=eps_smoothness,
        )
        if qq_df.empty:
            continue
        fig = plot_final_qq_grid(
            qq_df,
            sample_size=sample_size,
            rhos=QQ_RHOS,
            signal_smoothness=signal_smoothness,
            eps_smoothness=eps_smoothness,
            include_legend=True,
            include_x_axis=True,
            include_y_axis=True,
            include_title=True,
        )
        stem = (
            f"qq_calibration_N{sample_size}"
            f"_m_smoothness_{slugify(_format_smoothness(signal_smoothness))}"
            f"_eps_smoothness_{slugify(_format_smoothness(eps_smoothness))}"
            f"_rho_{'_'.join(slugify(_format_smoothness(rho)) for rho in QQ_RHOS)}"
            "_naive_vs_debiased"
        )
        for fmt in SAVE_FORMATS:
            path = FIGURE_DIR / f"{stem}.{fmt}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            final_qq_saved_paths.append(path)
        plt.close(fig)

print(f"Saved {len(final_qq_saved_paths)} final QQ figure files to {FIGURE_DIR}.")
if final_qq_saved_paths:
    display(pd.DataFrame({"path": [str(path) for path in final_qq_saved_paths]}))


## Paper tables 1–5 and denominator audit

The reported delta method is `delta_debiased_u`, reconstructed as the U-centered Wald interval for squared HSIC with standard error `2*sqrt(score_var/n)`, intersected with the nonnegative axis and mapped through square root. Every available repetition is retained, including negative U. Empty intervals have zero width and cover neither the target nor zero. The denominator audit must show zero exclusions for the reported methods. Error bars retain the historical clipped Wald binomial convention. Delta coverage is reported only for positive rho; null rows in the audit are descriptive.

Tables 1–5 are exported as CSV and LaTeX. `delta_coverage_update.csv` compares the archived conditional delta summaries with the squared-scale interval summaries using exactly the same source cohort. Point estimates, RTI intervals, and QQ plots are unchanged. QQ plots use squared-HSIC U-statistic errors centered at population HSIC and divided by the empirical sample standard deviation (ddof=1), preserving the current paper plots.


In [ ]:
PAPER_DELTA_METHOD = "delta_debiased_u"
PAPER_RTI_METHOD = "bootstrap_debiased"
PAPER_GROUPS = ["N", "rho", "f_smoothness", "eps_smoothness"]
if favourability.empty or coverage_all_ci.empty:
    raise ValueError("Paper coverage exports need nonempty input rows and confidence intervals")

provenance_columns = ["source_run_name", "run_name", "results_path", "random_key", "N", "rho", "f_smoothness", "eps_smoothness", "dimx", "dimy"]
favourability[[c for c in provenance_columns if c in favourability]].to_csv(
    TABLE_DIR / "analysis_input_provenance.csv.gz", index=False
)
input_counts = favourability.groupby(PAPER_GROUPS, dropna=False).size().rename("input_rows").reset_index()
valid_summary = summarize_binary(
    coverage_all_ci, ["method"] + PAPER_GROUPS, "coverage", "coverage"
).rename(columns={"runs": "included_rows"})
method_grid = pd.DataFrame({"method": METHOD_ORDER})
coverage_denominators = input_counts.merge(method_grid, how="cross").merge(
    valid_summary, on=["method"] + PAPER_GROUPS, how="left", validate="one_to_one"
)
coverage_denominators["included_rows"] = coverage_denominators["included_rows"].fillna(0).astype(int)
coverage_denominators["excluded_rows"] = coverage_denominators["input_rows"] - coverage_denominators["included_rows"]
if (coverage_denominators["excluded_rows"] < 0).any():
    raise ValueError("Repeated interval records inflate a method's denominator")
empty_counts = coverage_all_ci.assign(empty=coverage_all_ci["lower"] > coverage_all_ci["upper"]).groupby(
    ["method"] + PAPER_GROUPS, observed=True, dropna=False
)["empty"].sum().rename("empty_intervals").reset_index()
coverage_denominators = coverage_denominators.merge(empty_counts, on=["method"] + PAPER_GROUPS, how="left", validate="one_to_one")
coverage_denominators["empty_intervals"] = coverage_denominators["empty_intervals"].fillna(0).astype(int)
coverage_denominators["coverage_convention"] = np.where(coverage_denominators["excluded_rows"] == 0, "all_available_repetitions", "conditional_on_nonmissing_endpoints")
if coverage_denominators.loc[coverage_denominators["method"].isin([PAPER_RTI_METHOD, PAPER_DELTA_METHOD]), "excluded_rows"].ne(0).any():
    raise ValueError("The reported methods must retain every available repetition")
coverage_denominators.to_csv(TABLE_DIR / "coverage_denominator_audit.csv", index=False)

# Keep method names in machine-readable summaries, even where the paper uses short labels.
paper_coverage = coverage_denominators.loc[
    coverage_denominators["method"].isin([PAPER_RTI_METHOD, PAPER_DELTA_METHOD])
    & coverage_denominators["N"].isin(GRID_SAMPLE_SIZES)
    & coverage_denominators["f_smoothness"].isin(SIGNAL_SMOOTHNESS_VALUES)
    & coverage_denominators["eps_smoothness"].isin([.25, .5, .75])
    & coverage_denominators["rho"].between(0, .5)
].copy()
paper_coverage.to_csv(TABLE_DIR / "paper_coverage_summary.csv", index=False)

def require_paper_rows(frame: pd.DataFrame, expected: int, description: str) -> None:
    if len(frame) != expected or frame["coverage"].isna().any():
        raise ValueError(f"Incomplete {description}: expected {expected} populated rows, got {len(frame)}; inspect coverage_denominator_audit.csv")

for table_number, method, rhos, label in [
    (1, PAPER_RTI_METHOD, [0, .1, .2, .3, .4, .5], "tab:experiment_details_reverse_triangle_coverage_main"),
    (2, PAPER_DELTA_METHOD, [.1, .2, .3, .4, .5], "tab:experiment_details_delta_coverage_main"),
]:
    panel = paper_coverage.loc[
        (paper_coverage["method"] == method)
        & np.isclose(paper_coverage["f_smoothness"], 1)
        & np.isclose(paper_coverage["eps_smoothness"], .75)
        & paper_coverage["rho"].round(10).isin(rhos)
    ].copy()
    require_paper_rows(panel, len(rhos) * len(GRID_SAMPLE_SIZES), f"paper table {table_number}")
    panel["display"] = panel.apply(_format_coverage_error_bar, axis=1)
    table = panel.pivot(index="rho", columns="N", values="display").reindex(index=rhos, columns=GRID_SAMPLE_SIZES)
    table.columns = [f"$n={int(n)}$" for n in table.columns]
    table.index.name = r"$\rho$"
    table.to_csv(TABLE_DIR / f"paper_table_{table_number}.csv")
    table.to_latex(TABLE_DIR / f"paper_table_{table_number}.tex", escape=False, label=label,
                   caption="Coverage over all available repetitions with clipped Wald 95\\% error bars; delta intervals invert U-centered squared-HSIC intervals onto the norm scale.")

coverage_range_rows = []
for table_number, sample_size in enumerate(GRID_SAMPLE_SIZES, start=3):
    rows = []
    for sm in SIGNAL_SMOOTHNESS_VALUES:
        for se in [.25, .5, .75]:
            if np.isclose(sm, 1) and np.isclose(se, .75):
                continue
            row = {"f_smoothness": sm, "eps_smoothness": se}
            for method, name in [(PAPER_RTI_METHOD, "Reverse-triangle"), (PAPER_DELTA_METHOD, "Debiased delta method")]:
                panel = paper_coverage.loc[
                    (paper_coverage["method"] == method) & (paper_coverage["N"] == sample_size)
                    & np.isclose(paper_coverage["f_smoothness"], sm) & np.isclose(paper_coverage["eps_smoothness"], se)
                ]
                if method == PAPER_DELTA_METHOD:
                    panel = panel.loc[panel["rho"] > 0]
                require_paper_rows(panel, 5 if method == PAPER_DELTA_METHOD else 6, f"table {table_number}, smoothness {sm}/{se}, {method}")
                low, high = panel["coverage"].min(), panel["coverage"].max()
                row[name] = f"{low:.3f}--{high:.3f}"
                coverage_range_rows.append({"N": sample_size, "f_smoothness": sm, "eps_smoothness": se,
                                            "method": method, "coverage_min": low, "coverage_max": high,
                                            "min_included_rows": panel["included_rows"].min(),
                                            "max_included_rows": panel["included_rows"].max()})
            rows.append(row)
    table = pd.DataFrame(rows).set_index(["f_smoothness", "eps_smoothness"])
    table.to_csv(TABLE_DIR / f"paper_table_{table_number}.csv")
    table.to_latex(TABLE_DIR / f"paper_table_{table_number}.tex", escape=False,
                   label=f"tab:experiment_details_additional_coverage_summary_N{sample_size}",
                   caption=f"Ranges of coverage point estimates over all available repetitions at n={sample_size}; these are not confidence intervals.")
pd.DataFrame(coverage_range_rows).to_csv(TABLE_DIR / "paper_coverage_ranges.csv", index=False)
print("Saved paper Tables 1–5 and coverage_denominator_audit.csv; delta method:", PAPER_DELTA_METHOD)
display(paper_coverage.groupby("method")[["input_rows", "included_rows", "excluded_rows"]].agg(["min", "max"]))

# Compare old and new delta summaries without changing cohorts or nuisance fits.
archived_ci = build_ci_frame(archived_favourability)
archived_summary = summarize_binary(
    archived_ci.loc[archived_ci["method"].astype(str) == PAPER_DELTA_METHOD],
    PAPER_GROUPS, "coverage", "coverage"
).rename(columns={"runs": "archived_included_rows", "coverage": "archived_conditional_coverage"})
update = paper_coverage.loc[paper_coverage["method"] == PAPER_DELTA_METHOD].merge(
    archived_summary[PAPER_GROUPS + ["archived_included_rows", "archived_conditional_coverage"]],
    on=PAPER_GROUPS, how="left", validate="one_to_one"
)
update["coverage_change"] = update["coverage"] - update["archived_conditional_coverage"]
update.to_csv(TABLE_DIR / "delta_coverage_update.csv", index=False)

import hashlib
import json
validation = {
    "protocol": "U-centered squared-HSIC Wald interval with SE=2*sqrt(score_var/n), intersected with [0,infinity) then square-rooted; empty sets have width zero and no coverage",
    "source": str(BASE),
    "source_pattern": FAVOURABILITY_PATTERN,
    "source_rows": len(favourability),
    "reported_delta_method": PAPER_DELTA_METHOD,
    "reported_method_excluded_rows": int(paper_coverage["excluded_rows"].sum()),
    "historical_comparison": "delta_coverage_update.csv; new delta results intentionally replace the old conditional-coverage values",
    "source_hashes": {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in ["synthetic_paper_plots.ipynb", "paper_plot_utils.py"]},
}
(TABLE_DIR / "paper_table_validation.json").write_text(json.dumps(validation, indent=2) + chr(10))
